# Signall — LLM Agent Training with GRPO

Train a small LLM to act as a multi-armed bandit agent using **Unsloth** (efficient LoRA) and **HF TRL** (GRPO reinforcement learning).

The model learns to choose optimal arms by receiving rewards from the live OpenEnv environment on HuggingFace Spaces.

**Requirements:** Colab free tier (T4 GPU, ~15GB VRAM)

In [ ]:
# Install dependencies
!pip install -q unsloth
!pip install -q --no-deps trl
!pip install -q aiohttp datasets nest_asyncio

In [ ]:
import json
import re
import random
import asyncio
import aiohttp
import nest_asyncio
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

# Colab already runs an event loop; nest_asyncio lets TRL call asyncio.run() inside it
nest_asyncio.apply()

# Environment
BANDIT_API = "https://jonyeazel-cognitive-primitives-bandit.hf.space"
NUM_ARMS = 6
ROUNDS_PER_EPISODE = 25

# Model
MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512
LORA_RANK = 8

# Training (conservative for Colab free tier T4)
NUM_PROMPTS = 200
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 4
NUM_GENERATIONS = 4
LEARNING_RATE = 5e-6

In [ ]:
# Load model with Unsloth LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    use_gradient_checkpointing="unsloth",
)

print(f"Trainable params: {model.num_parameters(only_trainable=True):,}")

In [ ]:
# OpenEnv MCP client — single-call interface that always works over HTTP

async def pull_arm(session, source_id):
    """Pull a bandit arm via MCP. Returns reward (float)."""
    payload = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "pull_arm", "arguments": {"source_id": source_id}},
        "id": 1,
    }
    async with session.post(
        f"{BANDIT_API}/mcp",
        json=payload,
        headers={"Content-Type": "application/json"},
    ) as resp:
        data = await resp.json()
        result = json.loads(data["result"]["content"][0]["text"])
        return result


# Verify connection
async def _test():
    async with aiohttp.ClientSession() as s:
        for arm in range(NUM_ARMS):
            result = await pull_arm(s, arm)
            print(f"Arm {arm}: reward={result['reward']:.2f}")

await _test()

In [ ]:
# Prompt and action formatting

SYSTEM_PROMPT = (
    "You are a multi-armed bandit agent. You have 6 arms (0-5). "
    "Each arm has a hidden average reward. Maximize total reward over 25 rounds. "
    "Respond with ONLY a JSON object: {\"source_id\": <0-5>}"
)


def observation_to_prompt(obs):
    """Convert observation dict to chat messages."""
    round_num = obs.get("round", 0)
    total_score = obs.get("total_score", 0.0)
    last_reward = obs.get("reward", 0.0)
    last_arm = obs.get("source_id", 0)
    remaining = ROUNDS_PER_EPISODE - round_num

    if round_num == 0:
        content = f"Episode started. {remaining} rounds remaining. Choose an arm."
    else:
        content = (
            f"Round {round_num}/{ROUNDS_PER_EPISODE}. "
            f"Last pull: arm {last_arm} \u2192 {last_reward:.1f}. "
            f"Total: {total_score:.1f}. "
            f"Choose next arm."
        )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": content},
    ]


def parse_action(completion):
    """Extract action from model completion, with fallback."""
    try:
        match = re.search(r"\{[^}]+\}", completion)
        if match:
            action = json.loads(match.group())
            sid = int(action.get("source_id", 0))
            return {"source_id": max(0, min(5, sid))}
    except Exception:
        pass
    return {"source_id": random.randint(0, 5)}

In [ ]:
# Generate diverse training prompts

prompts = []
for _ in range(NUM_PROMPTS // ROUNDS_PER_EPISODE):
    # Episode start
    prompts.append({"prompt": observation_to_prompt(
        {"round": 0, "total_score": 0.0, "reward": 0.0, "source_id": 0}
    )})
    # Mid-episode states
    for r in range(1, ROUNDS_PER_EPISODE):
        prompts.append({"prompt": observation_to_prompt({
            "round": r,
            "total_score": random.uniform(r * 2, r * 8),
            "reward": random.uniform(1, 9),
            "source_id": random.randint(0, 5),
        })})

train_dataset = Dataset.from_list(prompts)
print(f"Dataset: {len(train_dataset)} prompts")

In [ ]:
# Reward functions

def format_reward_func(completions, **kwargs):
    """Reward valid JSON output format."""
    rewards = []
    for comp in completions:
        try:
            match = re.search(r"\{[^}]+\}", comp)
            if match:
                parsed = json.loads(match.group())
                sid = parsed.get("source_id")
                if isinstance(sid, int) and 0 <= sid <= 5:
                    rewards.append(0.2)
                else:
                    rewards.append(0.05)
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards


async def env_reward_func(completions, **kwargs):
    """Reward from the live Bandit environment via MCP on HuggingFace Spaces."""

    async def _score(completion):
        action = parse_action(completion)
        try:
            async with aiohttp.ClientSession() as session:
                result = await pull_arm(session, action["source_id"])
            reward = result.get("reward", 0.0)
            return max(0.0, min(1.0, reward / 10.0))  # normalize to [0, 1]
        except Exception:
            return 0.0

    return list(await asyncio.gather(*[_score(c) for c in completions]))

In [ ]:
# Configure GRPO trainer

training_args = GRPOConfig(
    output_dir="./bandit_agent",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_generations=NUM_GENERATIONS,
    max_completion_length=64,
    max_prompt_length=256,
    learning_rate=LEARNING_RATE,
    num_train_epochs=1,
    warmup_steps=10,
    beta=0.0,
    loss_type="grpo",
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=5,
    save_steps=50,
    use_vllm=False,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    reward_funcs=[format_reward_func, env_reward_func],
    reward_weights=[0.2, 0.8],
)

print("Trainer ready.")
print(f"Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION * NUM_GENERATIONS}")

In [ ]:
# Train
trainer.train()
print("Training complete.")

In [ ]:
# Save LoRA adapters
model.save_pretrained("./bandit_agent_lora")
tokenizer.save_pretrained("./bandit_agent_lora")
print("Saved to ./bandit_agent_lora")

In [ ]:
# Evaluate: measure trained agent's arm selection and reward

FastLanguageModel.for_inference(model)

async def evaluate(num_pulls=50):
    """Pull arms based on model's choices and measure vs optimal."""
    arm_counts = [0] * NUM_ARMS
    arm_rewards = [0.0] * NUM_ARMS

    async with aiohttp.ClientSession() as session:
        for i in range(num_pulls):
            # Generate a mid-game prompt
            prompt = observation_to_prompt({
                "round": random.randint(1, 20),
                "total_score": random.uniform(10, 100),
                "reward": random.uniform(1, 9),
                "source_id": random.randint(0, 5),
            })

            inputs = tokenizer.apply_chat_template(
                prompt, return_tensors="pt", add_generation_prompt=True
            ).to(model.device)

            outputs = model.generate(
                inputs,
                max_new_tokens=32,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

            text = tokenizer.decode(
                outputs[0][inputs.shape[1]:], skip_special_tokens=True
            )
            action = parse_action(text)
            arm = action["source_id"]

            # Pull via live API
            result = await pull_arm(session, arm)
            reward = result.get("reward", 0.0)

            arm_counts[arm] += 1
            arm_rewards[arm] += reward

            if (i + 1) % 10 == 0:
                avg = sum(arm_rewards) / (i + 1)
                print(f"  Pull {i+1}/{num_pulls}: avg reward={avg:.2f}")

    # Summary
    print(f"\nArm selection over {num_pulls} pulls:")
    total_reward = sum(arm_rewards)
    for arm in range(NUM_ARMS):
        avg = arm_rewards[arm] / max(arm_counts[arm], 1)
        pct = arm_counts[arm] / num_pulls * 100
        print(f"  Arm {arm}: chosen {arm_counts[arm]}x ({pct:.0f}%)  avg reward={avg:.1f}")

    avg_reward = total_reward / num_pulls
    # Optimal single-pull avg is ~7.1 (best arm mean)
    efficiency = (avg_reward / 7.1) * 100
    print(f"\nAvg reward: {avg_reward:.2f}  (efficiency: {efficiency:.0f}% vs optimal)")

await evaluate()